<!-- ARTHASAATHI-PROVENANCE -->
> ## ⚠️ This notebook is demo evidence, not the running system
>
> The logic below was migrated to **`ml/src/councils/risk/debt_trap.py`** and that module is what the application actually executes.
>
> **Role in the system:** Risk Council -> Debt Trap Agent
>
> This notebook is preserved because its stored outputs are the record of the original analysis. **Editing the cells below will not change the system's behaviour** — change the module and its tests instead.
>
> Tests for this logic live under `tests/`, and every agent is covered by the cross-cutting contract suite in `tests/common/test_contracts.py`.


Calculate available capital

1)check if income > mandatory_expenses
        a)income > mandatory_expenses
        b)income + emergency fund > mandatory_expenses
        c)income < mandatory_expenses

Input:

In [419]:

tc = {
    "monthly_income": 95000,
    "essential_expenses": 72000,
    "emergency_fund": 0,
    "expected_scenario": "Designed to test whether the advisor prioritizes overdue accounts over interest rates."
}

debts = [
    {
        "name": "HDFC Card",
        "debt_type": "credit_card",
        "outstanding_amount": 90000,
        "interest_rate": 42,
        "minimum_due": 12000
    },
    {
        "name": "Axis Card education loan",
        "debt_broad_type": "loan",
        "debt_type": "education_loan",
        "original_amount": 500000,
        "no_of_months_of_loan": 84,
        "outstanding_amount": 390000,
        "interest_rate": 10,
        "overdue_cycles": 7,
        "emi": 6500
    },
    {
        "name": "tv_emi",
        "debt_broad_type": "emi",
        "debt_type": "emi",
        "original_amount": 150000,
        "outstanding_amount": 85000,
        "interest_rate": 12,
        "overdue_cycles": 6,
        "emi": 4500
    },
    {
        "name": "personal_loan",
        "debt_broad_type": "loan",
        "debt_type": "personal_loan",
        "original_amount": 300000,
        "outstanding_amount": 260000,
        "interest_rate": 17,
        "overdue_cycles": 5,
        "emi": 9000
    }
]

In [420]:
def calculate_emi_arrears(
    emi,
    annual_rate,
    overdue_cycles
):
    monthly_rate = annual_rate / 1200

    arrears = 0

    for k in range(
        1,
        overdue_cycles + 1
    ):
        arrears += (
            emi
            * (1 + monthly_rate) ** k
        )

    return arrears

In [421]:
# Fallback annual interest rates (%) when debt's interest_rate is None
DEFAULT_RATES = {
    "credit_card":           36,
    "bnpl":                  24,
    "personal_loan":         15,
    "gold_loan":             12,
    "education_loan":        10,
    "car_loan":              10,
    "home_loan":              9,
    "consumer_durable_loan": 12,
    "other":                 15,
    "emi":                 15,
}

def get_effective_rate(debt):
    """Return (annual_rate, source) for a debt. Uses actual rate if provided."""
    if debt.get("interest_rate") is not None:
        return debt["interest_rate"], "actual"
    rate = DEFAULT_RATES.get(debt["debt_type"], DEFAULT_RATES["other"])
    return rate, "estimated"

In [422]:
# def calculate_available_budget(income,mandatory_expenses,emergency_fund):
#     if(income > mandatory_expenses):
#         return [{"net_income" : income - mandatory_expenses,"track" : "Surpluss"}]
#     elif(income + emergency_fund > mandatory_expenses):
#         return [{"net_income" : income - mandatory_expenses,"track" : "EmergencyFundused"}]
#     else:
#         return [{"net_income" : mandatory_expenses - income - emergency_fund,"track" : "Deficit"}]

# available_budget = calculate_available_budget(MONTHLY_INCOME,ESSENTIAL_EXPENSES,EMERGENCY_FUND)

Rank as per max loss

In [423]:
import math


EMI_DEBTS = {
    "home_loan",
    "personal_loan",
    "education_loan",
    "car_loan",
    "gold_loan",
    "consumer_durable_loan",
    "emi"
}


def get_effective_rate(debt):

    if debt.get("interest_rate") is not None:
        return debt["interest_rate"], "actual"

    return DEFAULT_RATES.get(
        debt["debt_type"],
        DEFAULT_RATES["other"]
    ), "estimated"


def calculate_available_budget(
    monthly_income,
    essential_expenses,
    emergency_fund
):
    return monthly_income - essential_expenses

def calculate_available_budget_emergency(
    monthly_income,
    essential_expenses,
    emergency_fund
):
    return monthly_income - essential_expenses + emergency_fund


def calculate_mandatory_payments(debts):

    total = 0

    for debt in debts:

        if debt["debt_type"] in EMI_DEBTS:
            total += debt.get("emi", 0)

        else:
            total += debt.get("minimum_due", 0)

    return total


def calculate_loss_if_not_paid(debt):

    rate, _ = get_effective_rate(debt)

    monthly_rate = rate / 1200

    overdue_cycles = debt.get(
        "overdue_cycles",
        0
    )

    if debt["debt_type"] in EMI_DEBTS:

        emi = debt["emi"]

        return (
            emi
            * ((1 + monthly_rate) ** (overdue_cycles + 1))
            - emi
        )

    minimum_due = debt["minimum_due"]

    return (
        debt["outstanding_amount"] * monthly_rate 
    )


def get_debt_bucket(debt):

    if debt["debt_type"] in EMI_DEBTS:
        return "emi"

    return "revolving"


def rank_debts(
    debts,
    monthly_income,
    essential_expenses,
    emergency_fund
):

    available_budget = calculate_available_budget(
        monthly_income,
        essential_expenses,
        emergency_fund
    )

    available_budget_emergency = calculate_available_budget_emergency(
        monthly_income,
        essential_expenses,
        emergency_fund
    )

    mandatory_payments = calculate_mandatory_payments(
        debts
    )

    ranked = []

    for debt in debts:

        loss = calculate_loss_if_not_paid(
            debt
        )

        ranked.append({
            **debt,
            "loss_if_not_paid": round(
                loss,
                2
            )
        })

    #
    # CASE A
    #
    if available_budget >= mandatory_payments:

        ranked.sort(
            key=lambda d: d["interest_rate"],
            reverse=True
        )

        return {
            "scenario": "income_sufficient",
            "ranked_debts": ranked
        }

    #
    # CASE B
    #
    if (
        available_budget
        + emergency_fund
        >= mandatory_payments
    ):

        ranked.sort(
            key=lambda d: d["interest_rate"],
            reverse=True
        )

        return {
            "scenario": "use_emergency_fund",
            "ranked_debts": ranked
        }

    #
    # CASE C
    #
    bucket_priority = {
        "emi": 0,
        "revolving": 1
    }

    ranked.sort(
        key=lambda d: (
            bucket_priority[
                get_debt_bucket(d)
            ],
            d["outstanding_amount"],
        )
    )

    return {
        "scenario": "cannot_cover_mandatory",
        "ranked_debts": ranked
    }

In [424]:
import copy


def allocate_payments(
    monthly_income,
    essential_expenses,
    emergency_fund,
    scenario,
    debts
):

    proposed_solution = {
        "remaining_income": 0,
        "remaining_emergency_fund": emergency_fund,
        "emergency_fund_used": 0,
        "debts": copy.deepcopy(debts)
    }

    available_income = (
        monthly_income
        - essential_expenses
    )

    mandatory_payments = (
        calculate_mandatory_payments(
            debts
        )
    )

    # --------------------------------------------------
    # Determine usable cash
    # --------------------------------------------------

    if scenario == "income_sufficient":

        emergency_used = 0

        usable_cash = available_income

    elif scenario == "use_emergency_fund":

        shortfall = max(
            mandatory_payments
            - available_income,
            0
        )

        emergency_used = min(
            shortfall,
            emergency_fund
        )

        usable_cash = (
            available_income
            + emergency_used
        )

        emergency_fund -= emergency_used

    else:  # cannot_cover_mandatory

        emergency_used = emergency_fund

        usable_cash = (
            available_income
            + emergency_fund
        )

        emergency_fund = 0

    remaining_cash = usable_cash

    # --------------------------------------------------
    # Initialize fields
    # --------------------------------------------------

    for debt in proposed_solution["debts"]:

        payment = (
            debt.get("emi")
            or debt.get("minimum_due")
            or 0
        )

        debt["required_payment"] = payment

        debt["mandatory_paid"] = 0

        debt["extra_payment"] = 0

        debt["total_paid"] = 0

        debt["shortfall"] = payment

    # --------------------------------------------------
    # STEP 1
    # Mandatory payments
    # --------------------------------------------------

    for debt in proposed_solution["debts"]:

        payment = debt["required_payment"]

        if remaining_cash <= 0:
            break

        allocated = min(
            payment,
            remaining_cash
        )

        debt["mandatory_paid"] = allocated

        debt["shortfall"] = (
            payment - allocated
        )

        debt["outstanding_amount"] = max(
            debt["outstanding_amount"]
            - allocated,
            0
        )

        # # reduce overdue count by one cycle
        # if (
        #     allocated >= payment
        #     and debt.get(
        #         "overdue_cycles"
        #     ) is not None
        # ):
        #     debt["overdue_cycles"] = max(
        #         debt.get(
        #             "overdue_cycles",
        #             0
        #         ) - 1,
        #         0
        #     )

        # remaining_cash -= allocated

        # reduce overdue count by one cycle
        if (
            allocated >= payment
            and debt.get(
                "overdue_cycles"
            ) is not None
        ):
            debt["overdue_cycles"] = 0

        remaining_cash -= allocated

        if allocated < payment:

            debt["shortfall"] = (
                payment - allocated
            )

            monthly_rate = (
                debt.get(
                    "interest_rate",
                    0
                ) / 100
            ) / 12

            debt["outstanding_amount"] *= (
                1 + monthly_rate
            )

            if (
                debt.get(
                    "overdue_cycles"
                ) is not None
            ):
                debt["overdue_cycles"] += 1
        else:

            if (
                debt.get(
                    "overdue_cycles"
                ) is not None
            ):
                debt["overdue_cycles"] = 0

        
    # --------------------------------------------------
    # STEP 2
    # Avalanche surplus allocation
    # Debts already arrive ranked
    # --------------------------------------------------
# --------------------------------------------------
# STEP 2
# Avalanche surplus allocation
# --------------------------------------------------

    if remaining_cash > 0:

        for debt in proposed_solution["debts"]:

            if remaining_cash <= 0:
                break

            outstanding = (
                debt["outstanding_amount"]
            )

            if outstanding <= 0:
                continue

            extra = min(
                remaining_cash,
                outstanding
            )

            debt["extra_payment"] = extra

            debt["outstanding_amount"] -= extra

            if (
                debt["outstanding_amount"]
                <= 0
            ):
                if (
                    debt.get(
                        "overdue_cycles"
                    ) is not None
                ):
                    debt["overdue_cycles"] = 0

            remaining_cash -= extra
    # --------------------------------------------------
    # Final totals
    # --------------------------------------------------

    for debt in proposed_solution["debts"]:

            debt["total_paid"] = (
                    debt["mandatory_paid"]
                    + debt["extra_payment"]
                )

            proposed_solution[
                "mandatory_payments_met"
            ] = all(
                debt["mandatory_paid"]
                >= debt["required_payment"]
                for debt in proposed_solution[
                    "debts"
                ]
            )

            proposed_solution[
                "remaining_income"
            ] = remaining_cash

            proposed_solution[
                "remaining_emergency_fund"
            ] = emergency_fund

            proposed_solution[
                "emergency_fund_used"
            ] = emergency_used

            proposed_solution[
                "mandatory_payments"
            ] = mandatory_payments

    return proposed_solution

In [425]:
# =========================
# CASE A : Income Sufficient
# =========================

case_a_1 = {
    "monthly_income": 150000,
    "essential_expenses": 70000,
    "emergency_fund": 15000,
    "expected_scenario": "income_sufficient"
}

case_a_2 = {
    "monthly_income": 100000,
    "essential_expenses": 40000,
    "emergency_fund": 10000,
    "expected_scenario": "income_sufficient"
}


# =========================
# CASE B : Emergency Fund Used
# =========================

case_b_1 = {
    "monthly_income": 80000,
    "essential_expenses": 70000,
    "emergency_fund": 15000,
    "expected_scenario": "use_emergency_fund"
}
# available=10000
# mandatory=15000
# available+ef=25000


case_b_2 = {
    "monthly_income": 82000,
    "essential_expenses": 70000,
    "emergency_fund": 5000,
    "expected_scenario": "use_emergency_fund"
}
# available=12000
# mandatory=15000
# available+ef=17000


# =========================
# CASE C : Cannot Cover Mandatory
# =========================

case_c_1 = {
    "monthly_income": 75000,
    "essential_expenses": 70000,
    "emergency_fund": 5000,
    "expected_scenario": "cannot_cover_mandatory"
}
# available=5000
# available+ef=10000
# mandatory=15000


case_c_2 = {
    "monthly_income": 70000,
    "essential_expenses": 65000,
    "emergency_fund": 2000,
    "expected_scenario": "cannot_cover_mandatory"
}
# available=5000
# available+ef=7000
# mandatory=15000


TEST_CASES = [
    case_a_1,
    case_a_2,
    case_b_1,
    case_b_2,
    case_c_1,
    case_c_2
]

In [426]:
from pprint import pprint


ranked = rank_debts(
        debts=debts,
        monthly_income=tc["monthly_income"],
        essential_expenses=tc["essential_expenses"],
        emergency_fund=tc["emergency_fund"]
    )

    

allocation = allocate_payments(
        monthly_income=tc["monthly_income"],
        essential_expenses=tc["essential_expenses"],
        emergency_fund=tc["emergency_fund"],
        scenario=ranked["scenario"],
        debts=ranked["ranked_debts"]
    )

print("\n" + "=" * 60)
print("TEST CASE")
print("=" * 60)
print("Expected :", tc["expected_scenario"])
print("Actual   :", ranked["scenario"])
print(
        "Original Income :",
        tc["monthly_income"]
    )
print(
        "Essential expenses :",
        tc["essential_expenses"]
    )

print(
        "Emergency Fund :",
        tc["emergency_fund"]
    )

print(
        "Remaining Income :",
        allocation["remaining_income"]
    )

print(
        "Remaining Emergency Fund :",
        allocation["remaining_emergency_fund"]
    )

print("\nDebt Allocations")

for debt in allocation["debts"]:

        print(
            f"{debt['name']:<30}"
            f"Mandatory: ₹{debt['mandatory_paid']:<8,.0f}"
            f"Extra: ₹{debt['extra_payment']:<8,.0f}"
            f"Total: ₹{debt['total_paid']:<8,.0f}"
            f"Outstanding: ₹{debt['outstanding_amount']:<10,.0f}"
            f"Overdue: {debt.get('overdue_cycles', 0)}"
        )


TEST 6
Expected : Designed to test whether the advisor prioritizes overdue accounts over interest rates.
Actual   : cannot_cover_mandatory
Original Income : 95000
Essential expenses : 72000
Emergency Fund : 0
Remaining Income : 0
Remaining Emergency Fund : 0

Debt Allocations
tv_emi                        Mandatory: ₹4,500   Extra: ₹0       Total: ₹4,500   Outstanding: ₹80,500    Overdue: 0
personal_loan                 Mandatory: ₹9,000   Extra: ₹0       Total: ₹9,000   Outstanding: ₹251,000   Overdue: 0
Axis Card education loan      Mandatory: ₹6,500   Extra: ₹0       Total: ₹6,500   Outstanding: ₹383,500   Overdue: 0
HDFC Card                     Mandatory: ₹3,000   Extra: ₹0       Total: ₹3,000   Outstanding: ₹90,045    Overdue: 0


In [427]:
from datetime import datetime, timedelta


def process_due_debts(debts):
    """
    Daily cron job (12:01 AM).

    Logic:
    1. Check if yesterday was the due date.
    2. Determine required payment.
    3. Determine shortfall.
    4. If shortfall exists:
         - Increment overdue cycle
         - Add shortfall to outstanding
         - Add monthly interest ONLY on shortfall
    5. Do not apply interest on the entire outstanding balance.
    """

    yesterday_day = (
        datetime.now() - timedelta(days=1)
    ).day

    for debt in debts:

        due_day = int(debt["date_of_payment"])

        if due_day != yesterday_day:
            continue

        required_payment = max(
            debt.get("minimum_due", 0),
            debt.get("emi", 0)
        )

        paid_amount = debt.get(
            "paid_this_cycle",
            0
        )

        shortfall = max(
            0,
            required_payment - paid_amount
        )

        if shortfall == 0:
            continue

        debt["overdue_cycles"] = (
            debt.get("overdue_cycles", 0) + 1
        )

        monthly_rate = (
            debt["interest_rate"]
            / 12
            / 100
        )

        penalty_interest = (
            shortfall * monthly_rate
        )

        debt["outstanding_amount"] += (
            shortfall +
            penalty_interest
        )

        debt["last_serviced"] = datetime.now().strftime(
            "%Y-%m-%d"
        )

        debt["last_shortfall"] = round(
            shortfall,
            2
        )

        debt["last_penalty_interest"] = round(
            penalty_interest,
            2
        )

    return debts


# --------------------------------------------------
# Example
# --------------------------------------------------

debts = [
    {
        "name": "HDFC Card",
        "debt_type": "credit_card",
        "outstanding_amount": 140000,
        "interest_rate": 42,
        "minimum_due": 18000,
        "paid_this_cycle": 0,
        "overdue_cycles": 0,
        "date_of_payment": "07"
    },
    {
        "name": "Axis Education Loan",
        "debt_type": "education_loan",
        "outstanding_amount": 310000,
        "interest_rate": 12,
        "emi": 15000,
        "paid_this_cycle": 14000,
        "overdue_cycles": 5,
        "date_of_payment": "07"
    }
]


updated = process_due_debts(debts)

for debt in updated:

    print("\n" + "=" * 60)

    print("Name:", debt["name"])

    print(
        "Outstanding:",
        round(
            debt["outstanding_amount"],
            2
        )
    )

    print(
        "Overdue Cycles:",
        debt.get(
            "overdue_cycles",
            0
        )
    )

    print(
        "Last Shortfall:",
        debt.get(
            "last_shortfall",
            0
        )
    )

    print(
        "Penalty Interest:",
        debt.get(
            "last_penalty_interest",
            0
        )
    )


Name: HDFC Card
Outstanding: 140000
Overdue Cycles: 0
Last Shortfall: 0
Penalty Interest: 0

Name: Axis Education Loan
Outstanding: 310000
Overdue Cycles: 5
Last Shortfall: 0
Penalty Interest: 0
